In [1]:
# ✅ Handwritten Digit Recognition (Gradio 5.49.1 compatible)
# Works in Colab or locally. No errors.

!pip install -q gradio==5.49.1 matplotlib tensorflow

import io, random
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

# -------------------------
# Prepare MNIST & Model
# -------------------------
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1, 28, 28, 1)

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=3, validation_data=(x_test, y_test), verbose=2)
_, test_acc = model.evaluate(x_test, y_test, verbose=0)
model_accuracy_text = f"Model accuracy (test set): {test_acc*100:.2f}%"

# -------------------------
# Utilities
# -------------------------
def preprocess_pil_for_model(pil_img: Image.Image):
    pil = pil_img.convert("L").resize((28, 28))
    arr = np.array(pil).astype(np.float32) / 255.0
    arr = 1.0 - arr
    return arr.reshape(1, 28, 28, 1)

def probs_to_hbar_image(probs):
    plt.figure(figsize=(6, 2.6))
    digits = np.arange(10)
    plt.barh(digits, probs, align='center')
    plt.yticks(digits, [str(d) for d in digits])
    plt.xlim(0, 1)
    plt.xlabel("Probability")
    plt.title("Prediction Confidence")
    for i, p in enumerate(probs):
        plt.text(p + 0.01, i, f"{p*100:.1f}%", va='center', fontsize=9)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=110, bbox_inches='tight', transparent=True)
    plt.close()
    buf.seek(0)
    return Image.open(buf).convert("RGBA")

def get_random_test_sample():
    idx = random.randrange(len(x_test))
    arr = (x_test[idx].reshape(28,28) * 255).astype(np.uint8)
    pil = Image.fromarray(arr).convert("L").resize((280,280), resample=Image.NEAREST)
    label = int(y_test[idx])
    return pil, label

# -------------------------
# Prediction Function
# -------------------------
def predict_and_report(mode_choice, uploaded_image, drawn_image, expected_text, auto_clear_flag):
    img = uploaded_image if mode_choice == "Upload" else drawn_image
    if img is None:
        return "<div style='text-align:center;font-size:56px;color:#aaa;'>-</div>", None, model_accuracy_text, "", gr.update()

    x = preprocess_pil_for_model(img)
    preds = model.predict(x, verbose=0)[0]
    pred_digit = int(np.argmax(preds))
    chart_img = probs_to_hbar_image(preds)

    big_html = f"""
    <div style="text-align:center;">
      <div style="font-size:88px; font-weight:800; color:#00c4a7; letter-spacing:6px; margin:6px 0;">
        {pred_digit}
      </div>
      <div style="text-align:center;color:#cfeee6;font-weight:600;">Top prediction</div>
    </div>
    """

    confetti_html = ""
    if expected_text:
        try:
            expected_int = int(float(expected_text))
            if expected_int == pred_digit and preds[pred_digit] >= 0.5:
                confetti_html = """
                <script src="https://cdn.jsdelivr.net/npm/canvas-confetti@1.5.1/dist/confetti.browser.min.js"></script>
                <script>
                (function(){
                  if (window.confetti) {
                    window.confetti({ particleCount: 140, spread: 140, origin: { y: 0.6 } });
                  }
                })();
                </script>
                """
        except:
            pass

    clear_updates = gr.update(value=None) if auto_clear_flag else gr.update()
    return big_html, chart_img, model_accuracy_text, confetti_html, clear_updates

# -------------------------
# CSS & Header HTML
# -------------------------
css = """
body { background: radial-gradient(circle at 10% 10%, #071327 0%, #04121a 50%, #02060a 100%); color: #e6fff9; font-family: 'Inter', system-ui; }
.gradio-container { max-width: 1100px !important; margin: 18px auto !important; }
.panel { background: rgba(255,255,255,0.03); border-radius: 14px; padding: 14px; border: 1px solid rgba(255,255,255,0.04); }
.btn { border-radius:10px; padding:8px 12px; background: linear-gradient(90deg,#ff8b6a,#ff5f7a); color:white; border:none; font-weight:600; }
"""

header_html = """
<div class='header'>
  <h1>🖊️ Handwritten Digit Recognition</h1>
  <p>Upload or draw a digit (0–9). The CNN predicts and shows confidence.</p>
</div>
"""

# -------------------------
# Build the App
# -------------------------
with gr.Blocks(css=css, title="MNIST Digit Recognition") as demo:
    gr.HTML(header_html)

    with gr.Row():
        with gr.Column(scale=7):
            gr.HTML("<div class='panel'><h3>Input</h3><p>Choose what input to use.</p>")

            mode = gr.Radio(["Upload", "Draw"], value="Upload", label="Input mode")
            uploaded = gr.Image(label="Upload image (optional)", type="pil")
            drawn = gr.Sketchpad(label="Draw digit (optional)", width=280, height=280)
            gr.HTML("</div>")


            with gr.Row():
                sample_btn = gr.Button("🎲 Load Random Sample", elem_classes="btn")
                clear_btn = gr.Button("🧽 Clear Inputs", elem_classes="btn")
                expected = gr.Textbox(label="Expected digit (0-9)", placeholder="e.g. 7")
                auto_clear = gr.Checkbox(label="Auto-clear after predict", value=False)
                predict_btn = gr.Button("🔍 Predict", elem_classes="btn")

        with gr.Column(scale=5):
            gr.HTML("<div class='panel'><h3>Output</h3><p>Prediction & confidence</p>")
            big_digit = gr.HTML("<div id='big-digit'>-</div>")
            conf_chart = gr.Image(label="Confidence Chart")
            acc_md = gr.Markdown(model_accuracy_text)
            confetti_out = gr.HTML("")
            gr.HTML("</div>")

    gr.HTML("<div class='footer'>Digit Recognition | By Pankaj & Team</div>")

    predict_btn.click(
        fn=predict_and_report,
        inputs=[mode, uploaded, drawn, expected, auto_clear],
        outputs=[big_digit, conf_chart, acc_md, confetti_out, uploaded]
    )

    def load_sample():
        pil, lbl = get_random_test_sample()
        return pil
    sample_btn.click(fn=load_sample, inputs=None, outputs=[uploaded])

    def do_clear():
        return None, None, "<div id='big-digit'>-</div>", None, None
    clear_btn.click(fn=do_clear, inputs=None, outputs=[uploaded, drawn, big_digit, conf_chart, confetti_out])

demo.launch()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.0 MB/s eta 0:00:00
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
1875/1875 - 50s - 27ms/step - accuracy: 0.9576 - loss: 0.1404 - val_accuracy: 0.9817 - val_loss: 0.0524
Epoch 2/3
1875/1875 - 81s - 43ms/step - accuracy: 0.9861 - loss: 0.0463 - val_accuracy: 0.9872 - val_loss: 0.0373
Epoch 3/3
1875/1875 - 46s - 25ms/step - accuracy: 0.9900 - loss: 0.0323 - val_accuracy: 0.9863 - val_loss: 0.0426
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3dc2c046923685fbae.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
